# Step 04 — semantic enrichment

Step 03 answered *where the buildings are and how big they are*. This notebook
answers **what each one is for**.

| | |
|---|---|
| **Reads** | `data/output/03_alkis_by_building.gpkg` (426 MB, 869,316 buildings), `data/output/01_all_buildings_osm.gpkg` (161 MB, 509,792 OSM footprints) and `data/output/01_all_pois.gpkg` (23,048 POIs, for the presence test) |
| **Writes** | `data/experimental_extract/04_buildings_labelled.gpkg` + `.qml` — for QGIS |
| **Needs** | `geopandas`, `pyogrio` |

## State of this notebook

Built **one step at a time**, each run and inspected before the next is written.

| step | | status |
|---|---|---|
| **04.1** | **Slim the layer** — 30 columns → 18, addresses combined | **implemented** |
| **04.2** | **Function labels + activity map** | **implemented** |
| **04.3** | **Fill the gaps with OSM footprints** — the buildings ALKIS does not have | **implemented** |
| **04.4** | **Drop what is not a place of activity** — 40 classes outright; 3 classes and the home-only buildings unless a POI sits inside | **implemented** |
| 04.5 | Join the OSM POI layer by `poi_role` | not written |
| 04.6 | Write the enriched layer | not written |

## Which input layer, and why

`03_alkis_by_building.gpkg` — **one row per ALKIS object**, not per LoD2 part.
Step 03 writes both and the part layer is the authoritative one, but for
semantics the building is the right unit: `function`, `name` and the address are
attributes of the ALKIS object, so every part of a building carries identical
values. Measured in step 03: **0 of 869,316 buildings have parts that disagree
on `function`.** Nothing semantic is lost by working at building level, and a
POI sitting inside a hospital belongs to the hospital rather than to whichever
wing happens to contain it.

The part layer is there to fall back on if a building-level answer ever looks
wrong.

In [1]:
import os, sys
from pathlib import Path

# --- locate the pipeline root -------------------------------------------------
# Same reason as steps 01-03: a notebook's working directory is not necessarily
# its own folder, so Path('..') is unreliable.
def _find_root(start):
    for d in (start, *start.parents):
        if (d / 'config.py').is_file() and (d / 'lib' / 'checks.py').is_file():
            return d
    return None

_nb_dir = Path(globals()['__vsc_ipynb_file__']).parent if '__vsc_ipynb_file__' in globals() else None
ROOT_DIR = _find_root(_nb_dir) if _nb_dir else None
ROOT_DIR = ROOT_DIR or _find_root(Path.cwd())
if ROOT_DIR is None:
    raise RuntimeError(
        'Cannot find the pipeline root (the folder containing config.py). '
        f'Looked upward from notebook dir {_nb_dir} and cwd {Path.cwd()}.'
    )
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

_share = Path(sys.prefix) / 'Library' / 'share'
if not _share.is_dir():
    _share = Path(sys.prefix) / 'share'
if (_share / 'gdal').is_dir():
    os.environ.setdefault('GDAL_DATA', str(_share / 'gdal'))
if (_share / 'proj').is_dir():
    os.environ.setdefault('PROJ_LIB', str(_share / 'proj'))

import time
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely

from config import (
    ALKIS_BY_BUILDING_FILE, ALKIS_SLIM_COLS, ALKIS_ADDRESS_PARTS, TARGET_CRS,
    BUILDING_FUNCTION_CODELIST_FILE, ALKIS_ACTIVITY_MAP_FILE, ALKIS_ACTIVITY_SEP,
    LABELLED_INSPECT_FILE, EXPERIMENTAL_DIR,
    ALL_BUILDINGS_OSM_FILE, LOD2_REGION_TILES_FILE,
    OSM_GAP_MAX_ALKIS_COVERAGE, OSM_GAP_EXCLUDE_BUILDING_TAGS,
    OSM_GAP_FUNCTION_CODE, OSM_GAP_LABEL_EN, OSM_GAP_AGS_MAX_DISTANCE_M,
    OSM_GAP_EXPECTED_SHARE_PCT,
    ALKIS_HOME_ONLY_ACTIVITIES, ALKIS_EXPECTED_HOME_ONLY,
    ALL_POIS_FILE, ALKIS_DROP_ALWAYS, ALKIS_DROP_UNLESS_POI, POI_SITE_RESCUE_MIN_AREA_M2,
)
from lib.checks import require_file, require_non_empty, require_crs, require_unique, require_cols

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

print('Root :', ROOT_DIR)
print('Input:', ALKIS_BY_BUILDING_FILE.name, '+', ALL_BUILDINGS_OSM_FILE.name, '+', ALL_POIS_FILE.name)

Root : C:\Users\Mayur Patel\Documents\GitHub\Capacity_Calculation-pipeline-FINAL
Input: 03_alkis_by_building.gpkg + 01_all_buildings_osm.gpkg + 01_all_pois.gpkg


## 1. Input contract

The building layer must be what step 03 promised: one row per `alkis_id`, flat
2D geometry in the target CRS, with `function` populated everywhere.

`function` being 100 % filled is the one that matters — it is the parameter the
whole classification turns on, and a gap there would propagate silently into
every activity assignment.

In [2]:
require_file(ALKIS_BY_BUILDING_FILE, 'ALKIS buildings (step 03)')

print('Reading (~20 s) ...', flush=True)
t0 = time.perf_counter()
bld = gpd.read_file(ALKIS_BY_BUILDING_FILE, layer='buildings')
print(f'  ok  {len(bld):,} rows x {len(bld.columns)} columns  '
      f'[{time.perf_counter() - t0:,.0f}s]')

require_non_empty(bld, 'buildings')
require_crs(bld, TARGET_CRS, 'buildings')
require_unique(bld, 'alkis_id', 'buildings')

if bld.geometry.has_z.any():
    raise AssertionError('geometry still carries Z - step 03 should have flattened it')
print(f'  ok  geometry is 2D: {bld.geometry.geom_type.value_counts().to_dict()}')

_nf = int(bld['function'].isna().sum())
if _nf:
    raise AssertionError(
        f'{_nf:,} buildings have no `function` code. Everything in this notebook '
        'keys off it, so a gap here would propagate silently.'
    )
print(f'  ok  function: 100 % filled, {bld["function"].nunique()} distinct codes')

  ok  03_alkis_by_building.gpkg (424.7 MB)
Reading (~20 s) ...


  ok  869,316 rows x 32 columns  [7s]
  ok  buildings: 869,316 rows
  ok  buildings: CRS EPSG:25832
  ok  buildings.alkis_id: unique and non-null (869,316)


  ok  geometry is 2D: {'MultiPolygon': 869316}
  ok  function: 100 % filled, 88 distinct codes


## 2. What arrived

Two things worth looking at before dropping anything.

**The function codes.** These are AdV `Gebaeudefunktion` values in the form
`<AAA object class>_<function>`. The prefix matters: `31001` is `AX_Gebaeude`,
an actual building, and the `51xxx` classes are other structures — canopies,
installations. Step 03 measured **11.57 % of rows are not `31001`**, which is
the single most important thing to know before joining the reference tables in
04.2: if they only cover `31001_*`, roughly 100,000 rows fall out of the join
with no error raised.

**Fill rates.** `name` and the address columns are sparse, and how sparse
decides what is possible later — an OSM name match can only reach the buildings
that have a name.

In [3]:
print('AAA object class prefixes:')
_pref = bld['function'].str.slice(0, 5).value_counts()
for p, c in _pref.items():
    print(f'  {p}   {c:>9,}   ({100 * c / len(bld):>5.2f} %)')
print(f'  -> not 31001: {len(bld) - _pref.get("31001", 0):,} '
      f'({100 * (1 - _pref.get("31001", 0) / len(bld)):.2f} %)')

print()
print('the 12 most common function codes:')
for f, c in bld['function'].value_counts().head(12).items():
    print(f'  {f:<14} {c:>9,}  ({100 * c / len(bld):>5.2f} %)')

print()
print('fill rates of the columns that are about to be kept or dropped:')
for c in sorted(bld.columns):
    if c == 'geometry':
        continue
    nn = bld[c].notna().sum()
    keep = 'keep' if c in ALKIS_SLIM_COLS else ('-> address' if c in ALKIS_ADDRESS_PARTS else 'DROP')
    print(f'  {c:<20} {100 * nn / len(bld):>6.2f} %  {str(bld[c].dtype):<8}  {keep}')

AAA object class prefixes:


  31001     768,756   (88.43 %)
  51009      94,582   (10.88 %)
  51002       3,669   ( 0.42 %)
  51003       1,988   ( 0.23 %)
  51001         254   ( 0.03 %)
  51006          50   ( 0.01 %)
  51007          17   ( 0.00 %)
  -> not 31001: 100,560 (11.57 %)

the 12 most common function codes:


  31001_2000       369,675  (42.52 %)
  31001_1000       327,264  (37.65 %)
  51009_1610        94,571  (10.88 %)
  31001_2720        18,850  ( 2.17 %)
  31001_2100        11,849  ( 1.36 %)
  31001_2010         8,631  ( 0.99 %)
  31001_2500         7,806  ( 0.90 %)
  31001_1120         5,793  ( 0.67 %)
  31001_1210         4,197  ( 0.48 %)
  31001_3000         3,301  ( 0.38 %)
  51003_1201         1,784  ( 0.21 %)
  51002_1250         1,772  ( 0.20 %)

fill rates of the columns that are about to be kept or dropped:
  ags                  100.00 %  str       keep
  alkis_id             100.00 %  str       keep
  area_m2              100.00 %  float64   keep
  city                 100.00 %  str       keep
  created_on           100.00 %  str       DROP
  elev_ground_min_m    100.00 %  float64   DROP
  elev_top_max_m       100.00 %  float64   DROP
  function             100.00 %  str       keep
  functions_all          0.00 %  object    DROP
  height_eaves_max_m   100.00 %  float64   keep

  roof_shape           100.00 %  str       keep


  street                34.91 %  str       -> address
  volume_3d_m3         100.00 %  float64   keep
  volume_old_m3        100.00 %  float64   keep
  volume_ratio         100.00 %  float64   keep


## 3. Combine the address, then slim to 18 columns

`street` and `house_number` become one `address` field — `"Aachener Straße 12"`
— falling back to the street alone when the number is missing, and to NULL when
neither exists.

**Expect it empty on about half the layer.** `street` and `house_number` are
~50 % filled, so anything downstream that wants address matching can only ever
reach half the building stock. That is a property of ALKIS, not of this step,
and it is better known now than discovered during a join.

What goes, and why, is recorded in `ALKIS_SLIM_COLS` in `config.py`. The two
worth restating here:

* **`n_functions` / `functions_all`** — always `1` and always NULL, because
  `function` belongs to the ALKIS object. They were guards for a disagreement
  that cannot occur in this region.
* **`elev_ground_min_m` / `elev_top_max_m`** — sea-level elevations. Capacity
  depends on heights above ground, which are kept. Note this drops the only
  topography signal in the layer; restore `elev_ground_min_m` if a later step
  wants terrain.

In [4]:
_street, _house = ALKIS_ADDRESS_PARTS
_s = bld[_street].fillna('').astype(str).str.strip()
_h = bld[_house].fillna('').astype(str).str.strip()
addr = (_s + ' ' + _h).str.strip()
bld['address'] = addr.where(addr != '', None)

print(f'  ..  {_street:<13} {100 * bld[_street].notna().mean():>6.2f} % filled')
print(f'  ..  {_house:<13} {100 * bld[_house].notna().mean():>6.2f} % filled')
print(f'  ..  address       {100 * bld["address"].notna().mean():>6.2f} % filled')
_street_only = int((bld[_street].notna() & bld[_house].isna()).sum())
print(f'  ..  street but no number: {_street_only:,}')
print()
print('  examples:')
for v in bld.loc[bld['address'].notna(), 'address'].head(5):
    print(f'      {v}')

missing = [c for c in ALKIS_SLIM_COLS if c not in bld.columns]
if missing:
    raise AssertionError(f'ALKIS_SLIM_COLS names columns that do not exist: {missing}')
dropped = [c for c in bld.columns if c not in ALKIS_SLIM_COLS]

slim = bld[ALKIS_SLIM_COLS].copy()
print()
print(f'  ..  {len(bld.columns)} columns -> {len(slim.columns)} '
      f'({len(dropped)} dropped)')
print(f'  ..  dropped: {", ".join(sorted(dropped))}')
require_unique(slim, 'alkis_id', 'slim buildings')

print()
print(slim.drop(columns='geometry').head(5).to_string(index=False))
print()
print('  the layer that goes into 04.2:')
for c in slim.columns:
    if c == 'geometry':
        continue
    nn = slim[c].notna().sum()
    print(f'      {c:<20} {100 * nn / len(slim):>6.2f} % filled   {slim[c].dtype}')

  ..  street         34.91 % filled
  ..  house_number   34.91 % filled
  ..  address        34.91 % filled
  ..  street but no number: 0

  examples:
      Ziegelofen 7
      Ackerweg 2
      Stieglitzweg 3
      Schulring 21
      Schaftrift 13

  ..  33 columns -> 19 (14 dropped)
  ..  dropped: created_on, elev_ground_min_m, elev_top_max_m, functions_all, house_number, n_functions, n_roof_clipped, n_roof_faces, n_surfaces, n_uuid, plan_acquired_on, roof_area_m2, roof_code, street


  ok  slim buildings.alkis_id: unique and non-null (869,316)

        alkis_id   function name        address                city      ags  area_m2  volume_3d_m3  volume_old_m3  volume_ratio  height_top_max_m  height_top_avg_m  height_eaves_max_m   roof_shape  roof_manual_any  roof_fallback_any  n_parts  is_multipart
DENIAL01000000Fg 51002_1250  NaN            NaN Braunschweig, Stadt 03101000     4.00          14.0           14.0        1.0000             3.500             3.500               3.500 PolyFlatRoof            False               True        1         False
DENIAL01000000Fh 51002_1250  NaN            NaN Braunschweig, Stadt 03101000     4.00          14.0           14.0        1.0000             3.500             3.500               3.500 PolyFlatRoof            False               True        1         False
DENIAL01000002A0 31001_1000  NaN   Ziegelofen 7 Braunschweig, Stadt 03101000   115.68         673.3          840.7        1.2486             7.404             7.268   

      address               34.91 % filled   str
      city                 100.00 % filled   str
      ags                  100.00 % filled   str
      area_m2              100.00 % filled   float64
      volume_3d_m3         100.00 % filled   float64
      volume_old_m3        100.00 % filled   float64
      volume_ratio         100.00 % filled   float64
      height_top_max_m     100.00 % filled   float64
      height_top_avg_m     100.00 % filled   float64
      height_eaves_max_m   100.00 % filled   float64
      roof_shape           100.00 % filled   str
      roof_manual_any      100.00 % filled   bool
      roof_fallback_any    100.00 % filled   bool
      n_parts              100.00 % filled   int64
      is_multipart         100.00 % filled   bool


## 4. Attach the labels and the activities

Two reference tables, both keyed on the same `31001_1000` form as the layer, so
both are plain left joins:

| file | rows | gives |
|---|---|---|
| `building_function_codelist_de_en.csv` | 301 | `label_en` — what the code *means* (the German `label_de` column is left in the file but not joined) |
| `alkis_building_activity_map.csv` | 280 | `activities` — what *happens* there |

The activity map is the consequential one: `activities` is what the
redistribution eventually weights, so a code missing from it yields a building
with no activity, which would drop out silently rather than raise anything.

**Measured coverage: 88 of 88 codes present in both.** Nothing falls out. The
check below asks the question in that direction anyway — *which of OUR codes are
missing?* — rather than the reassuring but useless "how much of the reference do
we use?".

The activity map was converted once from `alkis_building_activity_map.xlsx`
(kept beside it) so the pipeline needs no `openpyxl` and the table stays
greppable. Both CSVs carry a BOM, hence `utf-8-sig`: read as plain `utf-8` the
first column comes back with a `\ufeff` prefix and the join matches nothing.

The 14 atomic activities in use are `business, daycare, early_education,
education, errands, home, leisure, lessons, meetup, other, shopping, sports,
unspecified, work`. Note `unspecified` and `other` are real values in the
source, not placeholders — a building can legitimately end up with nothing
informative.

In [5]:
require_file(BUILDING_FUNCTION_CODELIST_FILE, 'AdV function codelist')
require_file(ALKIS_ACTIVITY_MAP_FILE, 'ALKIS activity map')

codes = pd.read_csv(BUILDING_FUNCTION_CODELIST_FILE, encoding='utf-8-sig', dtype=str)
acts = pd.read_csv(ALKIS_ACTIVITY_MAP_FILE, encoding='utf-8-sig', dtype=str)

# Both are keyed tables, so a duplicate key would quietly multiply rows in the
# join. Asserted, not assumed.
require_unique(codes, 'function', 'codelist')
if 'gfk_code' not in acts.columns:
    raise AssertionError(f'activity map columns are {list(acts.columns)}, '
                         "expected a 'gfk_code' column - check the BOM")
acts = acts.rename(columns={'gfk_code': 'function'})
require_unique(acts, 'function', 'activity map')
print(f'  ..  codelist {len(codes)} codes | activity map {len(acts)} codes')

# --- coverage, asked in the direction that can hurt -------------------------
ours = (slim.groupby('function')
        .agg(n=('alkis_id', 'size'), vol=('volume_3d_m3', 'sum')))
for tbl, name in ((codes, 'codelist'), (acts, 'activity map')):
    missing = ours[~ours.index.isin(set(tbl['function']))]
    if len(missing):
        print(f'  !!  {len(missing)} of our {len(ours)} codes are ABSENT from the '
              f'{name}: {missing["n"].sum():,} buildings '
              f'({100 * missing["n"].sum() / len(slim):.2f} %), '
              f'{missing["vol"].sum() / 1e6:,.1f}M m3 '
              f'({100 * missing["vol"].sum() / slim["volume_3d_m3"].sum():.2f} % of volume)')
        print(missing.sort_values('n', ascending=False).head(10).to_string())
    else:
        print(f'  ok  all {len(ours)} of our codes are present in the {name}')

# --- join --------------------------------------------------------------------
before = len(slim)
slim = slim.merge(codes[['function', 'label_en']], on='function', how='left')
slim = slim.merge(acts[['function', 'activities']], on='function', how='left')
if len(slim) != before:
    raise AssertionError(f'the joins changed the row count {before:,} -> {len(slim):,} '
                         '- a reference table has duplicate keys')
print(f'  ok  joined, still {len(slim):,} rows')

for c in ('label_en', 'activities'):
    nn = slim[c].notna().sum()
    print(f'  ..  {c:<12} {100 * nn / len(slim):>6.2f} % filled')

# The activity list, exploded, so it can be counted per atomic activity.
slim['n_activities'] = (slim['activities'].fillna('')
                        .str.split(ALKIS_ACTIVITY_SEP)
                        .map(lambda xs: len([x for x in xs if x.strip()])))
print()
print('  ..  buildings per atomic activity:')
_ex = (slim[['alkis_id', 'volume_3d_m3', 'activities']]
       .assign(a=slim['activities'].fillna('').str.split(ALKIS_ACTIVITY_SEP))
       .explode('a'))
_ex['a'] = _ex['a'].str.strip()
_ex = _ex[_ex['a'] != '']
_t = _ex.groupby('a').agg(buildings=('alkis_id', 'size'),
                          volume_Mm3=('volume_3d_m3', lambda v: v.sum() / 1e6))
_t['pct_vol'] = (100 * _t['volume_Mm3'] / (slim['volume_3d_m3'].sum() / 1e6)).round(2)
print(_t.sort_values('buildings', ascending=False).round(1).to_string())
_none = int(slim['n_activities'].eq(0).sum())
print(f'\n  ..  buildings with NO activity at all: {_none:,} '
      f'({100 * _none / len(slim):.3f} %)')

  ok  building_function_codelist_de_en.csv (0.0 MB)
  ok  alkis_building_activity_map.csv (0.0 MB)
  ok  codelist.function: unique and non-null (301)
  ok  activity map.function: unique and non-null (280)
  ..  codelist 301 codes | activity map 280 codes
  ok  all 88 of our codes are present in the codelist
  ok  all 88 of our codes are present in the activity map


  ok  joined, still 869,316 rows
  ..  label_en     100.00 % filled
  ..  activities   100.00 % filled



  ..  buildings per atomic activity:


                 buildings  volume_Mm3  pct_vol
a                                              
work                536702       366.7     54.1
business            402230       288.4     42.6
home                339246       324.2     47.9
meetup              338084       325.2     48.0
leisure             102430        32.1      4.7
errands              18986        62.9      9.3
shopping             15441        51.5      7.6
education             2710        17.3      2.6
lessons               1597         2.1      0.3
sports                 765         5.6      0.8
early_education        186         1.0      0.2
other                  105         0.1      0.0

  ..  buildings with NO activity at all: 0 (0.000 %)


## 5. Per-code summary — which building types are worth keeping?

Printed here for the discussion, not written to disk. This is the decision material, not a decision. One row per `function` code with
everything needed to judge it: what it is, what activities it has been assigned,
how many buildings, how much of the region's volume, and a physical profile
(median area, median height, share of flat roofs).

**The case that needs your eye most:**

```
51009_1610   Überdachung / canopy   →   activities: work;leisure
             94,571 buildings, 10.9 % of the layer
```

A canopy is a roof over a petrol forecourt or a bus stop. Nothing happens
*inside* it, because it has no inside — yet it carries `work;leisure`, so in a
volume-proportional redistribution those 94,571 structures compete with real
buildings for worker and leisure demand.

It is not obviously wrong to keep them: a covered loading bay at a depot is
arguably part of a workplace. But it is a decision, and the numbers below plus
the QGIS layer are what it should be made from rather than by inheriting whatever
the old pipeline did.

The `median_height` and `pct_flat_roof` columns are there because they separate
"structure" from "building" physically: a canopy is low and flat, a workshop is
not.

In [6]:
prof = slim.groupby(['function', 'label_en', 'activities'],
                    dropna=False).agg(
    n_buildings=('alkis_id', 'size'),
    volume_Mm3=('volume_3d_m3', lambda v: round(v.sum() / 1e6, 3)),
    median_area_m2=('area_m2', 'median'),
    median_height_m=('height_top_max_m', 'median'),
    median_volume_m3=('volume_3d_m3', 'median'),
    pct_flat_roof=('roof_shape', lambda s: round(100 * (s == 'PolyFlatRoof').mean(), 1)),
    pct_named=('name', lambda s: round(100 * s.notna().mean(), 2)),
).reset_index()
prof['pct_buildings'] = (100 * prof['n_buildings'] / len(slim)).round(3)
prof['pct_volume'] = (100 * prof['volume_Mm3'] /
                      (slim['volume_3d_m3'].sum() / 1e6)).round(3)
prof = prof.sort_values('n_buildings', ascending=False)

cols = ['function', 'label_en', 'activities', 'n_buildings', 'pct_buildings',
        'volume_Mm3', 'pct_volume', 'median_area_m2', 'median_height_m',
        'pct_flat_roof', 'pct_named']
print('the 20 codes with the most buildings:')
print(prof[cols].head(20).to_string(index=False))

print(f'\n  ..  {len(prof)} codes in total (printed here only - nothing is written)')

print()
print('  ..  the non-31001 classes, which are structures rather than buildings:')
_n31 = prof[~prof['function'].str.startswith('31001')]
print(_n31[cols].head(12).to_string(index=False))
print(f'\n      {len(_n31)} codes, {_n31["n_buildings"].sum():,} buildings '
      f'({100 * _n31["n_buildings"].sum() / len(slim):.2f} %), '
      f'{_n31["volume_Mm3"].sum():,.1f}M m3 '
      f'({100 * _n31["volume_Mm3"].sum() / (slim["volume_3d_m3"].sum() / 1e6):.2f} % of volume)')

the 20 codes with the most buildings:
  function                                                        label_en                                 activities  n_buildings  pct_buildings  volume_Mm3  pct_volume  median_area_m2  median_height_m  pct_flat_roof  pct_named
31001_2000                              Buildings for business or commerce                              work;business       369675         42.525     103.126      15.227          28.670           2.8730           74.3       0.05
31001_1000                                           residential buildings                                home;meetup       327264         37.646     299.945      44.287         109.030           8.5660           15.7       0.09
51009_1610                                                          canopy                               work;leisure        94571         10.879      10.098       1.491          10.100           3.8720           77.6       0.00
31001_2720                     Agricultural an

## 6. Fill the gaps in ALKIS with OSM footprints

ALKIS/LoD2 is the authoritative building stock, but it is not complete. OSM has
footprints where ALKIS has no record — buildings finished since the LoD2
release, sheds and huts the cadastre never took up, and a few hundred buildings
outside the LoD2 tile set altogether. Left alone, those are holes in the map and
in the redistribution. Here they are filled from `01_all_buildings_osm.gpkg`.

### How "no ALKIS record" is decided

By **footprint coverage**, not by a touch test: the share of each OSM
footprint's area that ALKIS buildings cover. A touch test would call an OSM shed
"present" because the neighbour's wall clips its corner by a few centimetres.

Measured, the coverage is sharply bimodal — most OSM footprints are either
untouched by ALKIS or more than half covered — and the threshold in
`OSM_GAP_MAX_ALKIS_COVERAGE` sits on the valley floor between the two modes.
The histogram is printed below so a re-run on other data shows whether the
valley is still where the threshold is. **The 10–50 % band is not filled**: an
OSM polygon a third covered by ALKIS is almost always the same building drawn
differently, and filling it would put a second footprint on top of the first.

### What the filled rows look like

They join the layer with the **same columns as the ALKIS rows**, plus:

| column | ALKIS rows | OSM rows |
|---|---|---|
| `source` | `alkis` | `osm` |
| `building_id` | the `alkis_id` | the OSM `bld_id` (`way/4708003`) — the one key that is unique across the whole layer |
| `function` | AdV code | **`OSM`** — a single synthetic class, so QGIS shows the whole fill as one legend entry |
| `label_en` | codelist | *Building mapped in OSM, no ALKIS record* |
| `osm_building` | NULL | the OSM `building=*` tag, for splitting the fill further |
| `osm_levels`, `osm_height_m` | NULL | `building:levels` / `height` where OSM has them (rare) |
| `ags`, `city` | ALKIS | from the **nearest ALKIS building**, so the administrative key stays in ALKIS's vocabulary |
| `area_m2` | ALKIS | the footprint area |
| `volume_*`, `height_*`, `roof_*` | ALKIS | **NULL** — there is no LoD2 model to measure |
| `activities` | activity map | **NULL** — no AdV code to map from |

Two consequences, both deliberate and both recorded in `config.py`:

* **No volume means zero weight** in a volume-proportional redistribution. These
  rows are on the map, not yet in the model. OSM has `building:levels` on about
  5 % of them — not enough to estimate the rest.
* **No activities means section 9 cannot judge them.** They all pass the
  home-only rule, including the ones tagged `house` and `apartments`. An OSM
  `building=*` → activity map is the obvious next reference table.


In [7]:
require_file(ALL_BUILDINGS_OSM_FILE, 'OSM buildings (step 01)')

print('Reading the OSM building layer ...', flush=True)
t0 = time.perf_counter()
osm = gpd.read_file(ALL_BUILDINGS_OSM_FILE, layer='buildings')
print(f'  ok  {len(osm):,} rows x {len(osm.columns)} columns  '
      f'[{time.perf_counter() - t0:,.0f}s]')
require_non_empty(osm, 'osm buildings')
require_crs(osm, TARGET_CRS, 'osm buildings')
require_unique(osm, 'bld_id', 'osm buildings')
require_cols(osm, ['bld_id', 'building', 'building_levels', 'height', 'name',
                   'addr_street', 'addr_housenumber'], 'osm buildings')
if not osm.geometry.is_valid.all():
    raise AssertionError(f'{int((~osm.geometry.is_valid).sum()):,} invalid OSM geometries '
                         '- the intersection below would fail on them')
osm['area_m2'] = osm.geometry.area

# --- how much of each OSM footprint does ALKIS cover? -------------------------
# Candidate pairs by bounding box, then the exact intersection area per pair,
# summed per OSM footprint. Two ALKIS buildings overlapping each other would
# count the same ground twice, so the share is capped at 1.
print('Measuring ALKIS coverage of every OSM footprint (~30 s) ...', flush=True)
t0 = time.perf_counter()
pairs = gpd.sjoin(osm[['geometry']], slim[['geometry']], how='inner', predicate='intersects')
_li = osm.index.get_indexer(pairs.index)
_ri = slim.index.get_indexer(pairs['index_right'])
_inter = shapely.area(shapely.intersection(osm.geometry.values[_li],
                                           slim.geometry.values[_ri]))
_covered = pd.Series(_inter).groupby(_li).sum()
osm['alkis_coverage'] = (
    _covered.reindex(range(len(osm))).fillna(0.0).to_numpy() / osm['area_m2'].to_numpy()
).clip(max=1.0)
print(f'  ok  {len(pairs):,} intersecting pairs; '
      f'{len(_covered):,} of {len(osm):,} OSM footprints touch an ALKIS building  '
      f'[{time.perf_counter() - t0:,.0f}s]')

print()
print('  ..  ALKIS coverage of the OSM footprints - the threshold should sit in the valley:')
_bins = [-np.inf, 0.0, 0.05, 0.10, 0.20, 0.50, 1.0]
_names = ['exactly 0', '0 - 0.05', '0.05 - 0.10', '0.10 - 0.20', '0.20 - 0.50', '0.50 - 1.00']
_h = pd.cut(osm['alkis_coverage'], _bins, labels=_names).value_counts().sort_index()
for k, v in _h.items():
    mark = '  <- below threshold' if k in _names[:3] else ''
    print(f'        {k:<12} {v:>9,}   ({100 * v / len(osm):>5.2f} %){mark}')
print(f'        threshold: coverage < {OSM_GAP_MAX_ALKIS_COVERAGE}')

# --- the gap -----------------------------------------------------------------
is_gap = osm['alkis_coverage'] < OSM_GAP_MAX_ALKIS_COVERAGE
excluded = osm['building'].isin(OSM_GAP_EXCLUDE_BUILDING_TAGS)
gap = osm[is_gap & ~excluded].copy()
_share = 100 * len(gap) / len(osm)
print()
print(f'  ..  gap: {len(gap):,} OSM footprints ALKIS does not have '
      f'({_share:.2f} % of OSM, {gap["area_m2"].sum() / 1e6:.2f} km2, '
      f'median {gap["area_m2"].median():.1f} m2)')
print(f'  ..  excluded {int((is_gap & excluded).sum()):,} tagged '
      f'building={sorted(OSM_GAP_EXCLUDE_BUILDING_TAGS)}')
_lo, _hi = OSM_GAP_EXPECTED_SHARE_PCT
if not (_lo <= _share <= _hi):
    print(f'  !!  {_share:.1f} % is outside the expected {_lo}-{_hi} % band - '
          'check the two layers share a CRS and that step 01 ran on the same PBF')
else:
    print(f'  ok  within the expected {_lo}-{_hi} % band')

# Coverage-boundary effect: where LoD2 has no tile, ALKIS cannot have a record,
# so those gaps are a missing tile set rather than missing buildings.
if LOD2_REGION_TILES_FILE.exists():
    _tiles = gpd.read_file(LOD2_REGION_TILES_FILE).to_crs(TARGET_CRS)
    _tile_union = shapely.union_all(_tiles.geometry.values)
    _no_tile = ~gap.geometry.representative_point().within(_tile_union)
    print(f'  ..  {int(_no_tile.sum()):,} of them lie outside every LoD2 tile '
          '(no ALKIS coverage there at all, not a missing record)')

print()
print('  ..  what OSM says they are (building=*):')
_bt = gap['building'].value_counts()
for tag, c in _bt.head(15).items():
    print(f'        {tag:<22} {c:>7,}   ({100 * c / len(gap):>5.2f} %)')
print(f'        ... {len(_bt) - 15} more tags')
print()
print('  ..  what OSM knows about them:')
for c in ('name', 'building_levels', 'height', 'addr_street'):
    print(f'        {c:<16} {100 * gap[c].notna().mean():>6.2f} % filled')

# --- administrative key from the nearest ALKIS building ---------------------
_nn = gpd.sjoin_nearest(gap[['geometry']], slim[['ags', 'city', 'geometry']],
                        how='left', max_distance=OSM_GAP_AGS_MAX_DISTANCE_M,
                        distance_col='_d')
_nn = _nn[~_nn.index.duplicated(keep='first')]     # equidistant ties
gap['ags'] = _nn['ags']
gap['city'] = _nn['city']
print()
print(f'  ..  ags/city from the nearest ALKIS building: '
      f'{gap["ags"].notna().sum():,} of {len(gap):,} matched within '
      f'{OSM_GAP_AGS_MAX_DISTANCE_M} m (median {_nn["_d"].median():.0f} m, '
      f'99 % within {_nn["_d"].quantile(0.99):.0f} m)')

# --- rows in the layer's own schema ------------------------------------------
_s = gap['addr_street'].fillna('').astype(str).str.strip()
_h = gap['addr_housenumber'].fillna('').astype(str).str.strip()
_addr = (_s + ' ' + _h).str.strip()
_height = pd.to_numeric(gap['height'].astype(str).str.extract(r'(\d+(?:[.,]\d+)?)')[0]
                        .str.replace(',', '.'), errors='coerce')

fill = gpd.GeoDataFrame({
    'building_id':  gap['bld_id'].to_numpy(),
    'source':       'osm',
    'alkis_id':     None,
    'function':     OSM_GAP_FUNCTION_CODE,
    'label_en':     OSM_GAP_LABEL_EN,
    'activities':   None,
    'n_activities': 0,
    'name':         gap['name'].to_numpy(),
    'address':      _addr.where(_addr != '', None).to_numpy(),
    'city':         gap['city'].to_numpy(),
    'ags':          gap['ags'].to_numpy(),
    'area_m2':      gap['area_m2'].round(2).to_numpy(),
    'n_parts':      1,
    'is_multipart': shapely.get_num_geometries(gap.geometry.values) > 1,
    'osm_building': gap['building'].to_numpy(),
    'osm_levels':   pd.to_numeric(gap['building_levels'], errors='coerce').to_numpy(),
    'osm_height_m': _height.to_numpy(),
    'geometry':     gap.geometry.values,
}, crs=TARGET_CRS)

slim['building_id'] = slim['alkis_id']
slim['source'] = 'alkis'
slim['osm_building'] = None
slim['osm_levels'] = np.nan
slim['osm_height_m'] = np.nan

fill = fill.reindex(columns=slim.columns)      # NULL for everything only ALKIS has
n_alkis = len(slim)
slim = gpd.GeoDataFrame(pd.concat([slim, fill], ignore_index=True),
                        geometry='geometry', crs=TARGET_CRS)
# Flags that are bool on ALKIS rows and NULL on OSM rows. Nullable boolean rather
# than object, so the GeoPackage gets 1/0/NULL and not the strings 'True'/'False'.
for c in ('roof_manual_any', 'roof_fallback_any', 'is_multipart'):
    slim[c] = slim[c].astype('boolean')
slim['n_parts'] = slim['n_parts'].astype('int64')

require_unique(slim, 'building_id', 'combined layer')
print(f'  ok  {n_alkis:,} ALKIS + {len(fill):,} OSM = {len(slim):,} buildings, '
      f'{len(slim.columns)} columns')
print()
print('  ..  the filled rows, as they now sit in the layer:')
_show = ['building_id', 'source', 'function', 'osm_building', 'name', 'address',
         'city', 'area_m2', 'osm_levels', 'volume_3d_m3', 'activities']
print(slim.loc[slim['source'] == 'osm', _show].head(6).to_string(index=False))


  ok  01_all_buildings_osm.gpkg (160.7 MB)
Reading the OSM building layer ...


  ok  509,792 rows x 23 columns  [3s]
  ok  osm buildings: 509,792 rows
  ok  osm buildings: CRS EPSG:25832
  ok  osm buildings.bld_id: unique and non-null (509,792)
  ok  osm buildings: has ['bld_id', 'building', 'building_levels', 'height', 'name', 'addr_street', 'addr_housenumber']


Measuring ALKIS coverage of every OSM footprint (~30 s) ...


  ok  867,259 intersecting pairs; 466,261 of 509,792 OSM footprints touch an ALKIS building  [31s]

  ..  ALKIS coverage of the OSM footprints - the threshold should sit in the valley:
        exactly 0       43,531   ( 8.54 %)  <- below threshold
        0 - 0.05         2,287   ( 0.45 %)  <- below threshold
        0.05 - 0.10      1,447   ( 0.28 %)  <- below threshold
        0.10 - 0.20      2,339   ( 0.46 %)
        0.20 - 0.50     17,047   ( 3.34 %)
        0.50 - 1.00    443,141   (86.93 %)
        threshold: coverage < 0.1

  ..  gap: 47,264 OSM footprints ALKIS does not have (9.27 % of OSM, 5.02 km2, median 33.3 m2)
  ..  excluded 1 tagged building=['no']
  ok  within the expected 5.0-15.0 % band


  ..  829 of them lie outside every LoD2 tile (no ALKIS coverage there at all, not a missing record)

  ..  what OSM says they are (building=*):
        yes                     33,603   (71.10 %)
        house                    3,088   ( 6.53 %)
        garage                   2,178   ( 4.61 %)
        shed                     1,807   ( 3.82 %)
        detached                 1,365   ( 2.89 %)
        hut                        887   ( 1.88 %)
        apartments                 531   ( 1.12 %)
        residential                469   ( 0.99 %)
        roof                       444   ( 0.94 %)
        semidetached_house         405   ( 0.86 %)
        carport                    318   ( 0.67 %)
        service                    198   ( 0.42 %)
        allotment_house            185   ( 0.39 %)
        bungalow                   184   ( 0.39 %)
        greenhouse                 170   ( 0.36 %)
        ... 74 more tags

  ..  what OSM knows about them:
        name               1.86


  ..  ags/city from the nearest ALKIS building: 47,167 of 47,264 matched within 1000 m (median 7 m, 99 % within 416 m)


  ok  combined layer.building_id: unique and non-null (916,580)
  ok  869,316 ALKIS + 47,264 OSM = 916,580 buildings, 27 columns

  ..  the filled rows, as they now sit in the layer:


 building_id source function osm_building                    name    address                city  area_m2  osm_levels  volume_3d_m3 activities
way/22943903    osm      OSM          yes                     NaN        NaN Braunschweig, Stadt    73.30         NaN           NaN       None
way/23862345    osm      OSM   commercial Hannes Camper Helmstedt Am Lohen 6    Helmstedt, Stadt   318.54         NaN           NaN       None
way/24989328    osm      OSM          yes                     NaN        NaN    Helmstedt, Stadt    97.75         NaN           NaN       None
way/25046958    osm      OSM          yes                     NaN        NaN      Süpplingenburg    78.40         NaN           NaN       None
way/25142110    osm      OSM          yes                     EU8        NaN    Wolfsburg, Stadt  1854.48         NaN           NaN       None
way/25142124    osm      OSM          yes                    EU10        NaN    Wolfsburg, Stadt   953.17         NaN           NaN       None

## 7. Which buildings hold a POI, and the drop rules

Two facts decide whether a building survives this notebook: **what ALKIS says it
is**, and **whether OSM knows of anything happening inside it**. Both are
computed here, before the export, so the map can show every decision.

### A POI "inside" a building

The three `poi_role`s are different kinds of evidence and are treated
differently:

* **`point`** and **`footprint`** — a node, or an OSM building polygon carrying
  the POI tag. Reduced to one point (the node, or the footprint's representative
  point) and tested for containment against every footprint in the layer, OSM
  fill included. A hit is evidence for *this* building: the median building it
  rescues is about 200 m², a real shop or practice, and the handful under 50 m²
  are holiday chalets that are the POI themselves. A point exactly on a shared
  wall is credited to one building only.
* **`site`** — an area polygon: a school's grounds, a care home, a campus, a
  riding centre, a holiday park. It says "activity somewhere in here", not "in
  this shed". Taken literally it would rescue the 285 garden huts of 1–3 m² in
  one allotment colony and the bike sheds on every school site. So a site
  rescues only the **substantial** buildings inside it: representative point
  within the site polygon *and* footprint at least
  `POI_SITE_RESCUE_MIN_AREA_M2` — bigger than any single-family house, smaller
  than a care-home wing. What the threshold keeps at other values is printed
  below so it can be moved.

`n_poi` counts point and footprint hits per building, `in_site` marks a building
inside a site polygon, and `has_poi` is *n_poi > 0, or in a site and at least
the threshold*. Whether a POI is a supermarket, a vacant unit or a viewpoint is
for the classification later, not for a filter here.

This is a *presence* test, not the POI join. 04.5 does the join properly — sites
onto every contained building, footprints 1:1, nested units — and snaps
building-bound POIs that lie outside every footprint. POIs inside a dropped
structure are ignored.

### The three rules, decided 2026-09-10

| rule | applies to | POI rescue |
|---|---|---|
| **1. class, always** | the 40 codes in `ALKIS_DROP_ALWAYS` — structures with no inside (canopy, mast, silo, tank, chimney, wind turbine, solar array, …) and building classes decided against (supply, disposal, greenhouses, transport operations, barracks, towers, stands, shelters, huts, ruins, mills, mines) | **none** — and the POIs inside are ignored, not re-homed |
| **2. class, unless a POI** | the 3 codes in `ALKIS_DROP_UNLESS_POI` — parking garages and decks, agricultural business buildings | yes — an Aldi in a parking garage, a riding stable or a farm shop stays |
| **3. home-only** | activities ⊆ `{home, meetup}` | yes — ALKIS codes a block by its dominant use, so the corner restaurant or the doctor's practice inside a residential building stays |

Precedence is top to bottom. `drop_reason` records which rule fired — `class`,
`class_no_poi`, `home_no_poi`, or NULL for kept — `kept` is its negation, and
`rescued` marks the buildings a POI saved from rule 2 or 3, with `rescued_by`
saying whether a `poi` inside or a `site` around it did the saving.

**The OSM fill is exempt by construction.** Its rows carry the synthetic
`function = 'OSM'` and no activities, so no class list and no activity rule can
match them. They pass through untouched, which section 9 reports.

Two guards. Every code in the two lists must exist in the codelist — a typo
would otherwise be a silent no-op, which is exactly the bug the old pipeline
shipped with — and the lists must not overlap. Any `51xxx` structure class
present in the data but in neither list is flagged, because on new data that is
almost certainly an omission.


In [8]:
# --- 7a. the lists must be typo-free and disjoint -----------------------------
_known = set(codes['function'])
for _nm, _lst in (('ALKIS_DROP_ALWAYS', ALKIS_DROP_ALWAYS),
                  ('ALKIS_DROP_UNLESS_POI', ALKIS_DROP_UNLESS_POI)):
    _bad = sorted(set(_lst) - _known)
    if _bad:
        raise AssertionError(f'{_nm} names codes that are not in the codelist - '
                             f'a typo here is a silent no-op: {_bad}')
_both = sorted(set(ALKIS_DROP_ALWAYS) & set(ALKIS_DROP_UNLESS_POI))
if _both:
    raise AssertionError(f'codes in both drop lists: {_both}')
_present = set(slim['function'])
print()
print(f'  ok  {len(ALKIS_DROP_ALWAYS)} + {len(ALKIS_DROP_UNLESS_POI)} codes in the drop lists, '
      f'all in the codelist, disjoint; {len(_present & set(ALKIS_DROP_ALWAYS))} + '
      f'{len(_present & set(ALKIS_DROP_UNLESS_POI))} of them occur in this region')

slim['aaa_class'] = slim['function'].str.split('_').str[0]
_unruled = [c for c in sorted(_present - set(ALKIS_DROP_ALWAYS) - set(ALKIS_DROP_UNLESS_POI))
            if c != OSM_GAP_FUNCTION_CODE and not c.startswith('31001_')]
if _unruled:
    print(f'  !!  {len(_unruled)} structure classes (not 31001) have no rule and will be '
          f'KEPT - almost certainly an omission: {_unruled}')
else:
    print('  ok  every non-31001 structure class present has a rule')

_acts = (slim['activities'].fillna('')
         .str.split(ALKIS_ACTIVITY_SEP)
         .map(lambda xs: frozenset(x.strip() for x in xs if x.strip())))
slim['home_only'] = _acts.map(lambda a: bool(a) and a <= ALKIS_HOME_ONLY_ACTIVITIES)
_always = slim['function'].isin(ALKIS_DROP_ALWAYS)
_unless = slim['function'].isin(ALKIS_DROP_UNLESS_POI)

# --- 7b. points and footprints: one point per POI, tested against every footprint
require_file(ALL_POIS_FILE, 'OSM POIs (step 01)')
pois = gpd.read_file(ALL_POIS_FILE, layer='pois')
require_non_empty(pois, 'pois')
require_crs(pois, TARGET_CRS, 'pois')
require_unique(pois, 'poi_id', 'pois')
require_cols(pois, ['poi_id', 'poi_role', 'poi_use', 'name', 'area_m2'], 'pois')
print()
print('  ok  ' + f'{len(pois):,} POIs: '
      + ', '.join(f'{k} {v:,}' for k, v in pois['poi_role'].value_counts().items()))

_pf = pois.loc[pois['poi_role'].isin(['point', 'footprint']),
               ['poi_id', 'poi_role', 'poi_use', 'name', 'geometry']].copy()
_pf['geometry'] = _pf.geometry.representative_point()
poi_hits = gpd.sjoin(_pf, slim[['building_id', 'function', 'source', 'geometry']],
                     how='inner', predicate='within')
poi_hits = poi_hits[~poi_hits.index.duplicated(keep='first')]   # a point on a shared wall: one building
_per_bld = poi_hits.groupby('building_id').size()
slim['n_poi'] = slim['building_id'].map(_per_bld).fillna(0).astype('int64')
print(f'  ..  {len(poi_hits):,} of {len(_pf):,} point/footprint POIs '
      f'({100 * len(poi_hits) / len(_pf):.1f} %) fall inside a footprint, '
      f'in {int((slim["n_poi"] > 0).sum()):,} buildings')
for _src, _g in slim.groupby('source'):
    print(f'        {_src:<6} {int((_g["n_poi"] > 0).sum()):>7,} buildings hold a POI  '
          f'({int(_g["n_poi"].sum()):,} POIs)')
_out = _pf.loc[~_pf['poi_id'].isin(poi_hits['poi_id']), 'poi_role'].value_counts()
print(f'  ..  outside every footprint: {int(_out.sum()):,}, by role: '
      + ', '.join(f'{k} {v:,}' for k, v in _out.items()))

# --- 7c. sites: only the substantial buildings inside an area POI -------------
_sites = pois.loc[pois['poi_role'] == 'site', ['poi_id', 'poi_use', 'name', 'geometry']]
_rep = slim[['building_id', 'geometry']].copy()
_rep['geometry'] = _rep.geometry.representative_point()
site_hits = gpd.sjoin(_rep, _sites, how='inner', predicate='within')
site_hits = site_hits[~site_hits.index.duplicated(keep='first')]   # nested sites: one is enough
slim['in_site'] = slim['building_id'].isin(site_hits['building_id'])
slim['has_poi'] = (slim['n_poi'] > 0) | (slim['in_site']
                                         & (slim['area_m2'] >= POI_SITE_RESCUE_MIN_AREA_M2))
print()
print(f'  ..  {len(_sites):,} site polygons contain {int(slim["in_site"].sum()):,} buildings; '
      f'{int((slim["in_site"] & (slim["n_poi"] == 0)).sum()):,} of them have no '
      'point/footprint POI of their own')
_cand = slim['in_site'] & (slim['n_poi'] == 0) & ~_always & (_unless | slim['home_only'])
print(f'  ..  {int(_cand.sum()):,} of those would be dropped by rule 2 or 3 - what a site '
      'ALONE rescues, by minimum footprint:')
for _t in (0, 30, 50, 100, 200, 500):
    _m = _cand & (slim['area_m2'] >= _t)
    _mark = '  <- POI_SITE_RESCUE_MIN_AREA_M2' if _t == POI_SITE_RESCUE_MIN_AREA_M2 else ''
    print(f'        >= {_t:>3} m2   {int(_m.sum()):>5,} buildings   '
          f'{slim.loc[_m, "volume_3d_m3"].sum() / 1e6:>5.2f}M m3{_mark}')
_big = _cand & (slim['area_m2'] >= POI_SITE_RESCUE_MIN_AREA_M2)
_bs = site_hits.loc[site_hits['building_id'].isin(slim.loc[_big, 'building_id']), 'poi_use']
print('      the sites doing the rescuing: '
      + ', '.join(f'{k} {v}' for k, v in _bs.value_counts().head(8).items()))

# --- 7d. the three rules, in order of precedence ------------------------------
slim['drop_reason'] = np.select(
    [_always, _unless & ~slim['has_poi'], slim['home_only'] & ~slim['has_poi']],
    ['class', 'class_no_poi', 'home_no_poi'],
    default=None)
slim['kept'] = slim['drop_reason'].isna()
slim['rescued'] = (_unless | slim['home_only']) & slim['has_poi'] & ~_always
slim['rescued_by'] = np.select(
    [slim['rescued'] & (slim['n_poi'] > 0), slim['rescued']],
    ['poi', 'site'],
    default=None)

print()
print('  ..  what the rules decide (applied and reported in section 9):')
_t = (slim.groupby(slim['drop_reason'].fillna('kept'))
      .agg(buildings=('building_id', 'size'),
           volume_Mm3=('volume_3d_m3', lambda v: round(v.sum() / 1e6, 1)),
           pois_inside=('n_poi', 'sum')))
print(_t.reindex(['class', 'class_no_poi', 'home_no_poi', 'kept']).to_string())
print(f'  ..  rescued: {int(slim["rescued"].sum()):,} buildings that rule 2 or 3 would '
      f'otherwise have dropped - {int((slim["rescued_by"] == "poi").sum()):,} by a POI inside, '
      f'{int((slim["rescued_by"] == "site").sum()):,} by a site polygon around them')



  ok  40 + 3 codes in the drop lists, all in the codelist, disjoint; 40 + 3 of them occur in this region


  ok  every non-31001 structure class present has a rule


  ok  01_all_pois.gpkg (11.3 MB)


  ok  pois: 23,048 rows
  ok  pois: CRS EPSG:25832
  ok  pois.poi_id: unique and non-null (23,048)
  ok  pois: has ['poi_id', 'poi_role', 'poi_use', 'name', 'area_m2']

  ok  23,048 POIs: point 12,706, footprint 9,471, site 871


  ..  18,199 of 22,177 point/footprint POIs (82.1 %) fall inside a footprint, in 14,358 buildings
        alkis   13,857 buildings hold a POI  (17,612 POIs)
        osm        501 buildings hold a POI  (587 POIs)
  ..  outside every footprint: 3,978, by role: point 2,048, footprint 1,930



  ..  871 site polygons contain 11,688 buildings; 10,840 of them have no point/footprint POI of their own
  ..  1,179 of those would be dropped by rule 2 or 3 - what a site ALONE rescues, by minimum footprint:
        >=   0 m2   1,179 buildings    2.10M m3
        >=  30 m2     910 buildings    2.09M m3
        >=  50 m2     795 buildings    2.07M m3
        >= 100 m2     548 buildings    1.99M m3
        >= 200 m2     322 buildings    1.78M m3  <- POI_SITE_RESCUE_MIN_AREA_M2
        >= 500 m2     106 buildings    1.25M m3
      the sites doing the rescuing: social_facility 79, school 29, university 22, hospital 21, equestrian 17, factory 17, chalet 16, research_institute 15

  ..  what the rules decide (applied and reported in section 9):


              buildings  volume_Mm3  pois_inside
drop_reason                                     
class            110674        25.3          233
class_no_poi      18861        29.8            0
home_no_poi      327113       289.7            0
kept             459932       332.5        17966
  ..  rescued: 4,482 buildings that rule 2 or 3 would otherwise have dropped - 4,160 by a POI inside, 322 by a site polygon around them


## 8. The class-by-class review export

Every building in the layer, labelled — the 869,316 from ALKIS **and the OSM
footprints section 6 added**, **including the ones section 9 drops** — so the
decision can be made by looking rather than from counts alone.

GeoPackage rather than shapefile: `label_en` and `activities` are long strings
and shapefile truncates field names to 10 characters.

### Stepping through the classes

A **`.qml` style file** is written beside the layer, so QGIS loads it already
categorised by `function` with all 89 classes in the legend — the 88 AdV codes
and the `OSM` fill. Each legend entry
can be ticked on and off individually — that is the class-by-class review,
without typing a single filter expression.

Colours are grouped by what the code family means, so the map is readable before
you touch anything:

| family | colour |
|---|---|
| `31001_1xxx` residential | blues |
| `31001_2xxx` business, commerce | oranges and reds |
| `31001_3xxx` public purposes | greens |
| `51xxx` other structures | greys |
| `OSM` footprints ALKIS does not have | magenta |

Legend entries read `31001_1000 · residential buildings · 327,264` so the label
carries the code, the meaning and the size without a lookup.

Ten extra columns make manual filtering easy where you do want it:

* **`class_label`** — the same `code · meaning · count` string, if you would
  rather categorise on that than on the bare code
* **`aaa_class`** — `31001` or `51009` etc., for splitting buildings from
  structures in one filter
* **`kept`** — what section 9 removes, as one flag. Styling by it shows the
  drop directly.
* **`drop_reason`** — *which* rule removed it: `class` (rule 1, outright),
  `class_no_poi` (rule 2, no POI inside), `home_no_poi` (rule 3, no POI inside),
  NULL for kept
* **`rescued`** — true where a POI saved a building rule 2 or 3 would have
  dropped: the residential block with the restaurant, the parking garage with
  the supermarket, the care-home wing coded residential
* **`rescued_by`** — `poi` when a point or footprint POI sits inside the
  building, `site` when only a site polygon around it did the saving (and the
  building is at least `POI_SITE_RESCUE_MIN_AREA_M2`)
* **`n_poi`** — how many point/footprint POIs fall inside the footprint, and
  **`in_site`** — whether the building lies inside a site polygon at all
* **`source`** — `alkis` or `osm`, and **`osm_building`** — the OSM `building=*`
  tag on the filled rows, for splitting the magenta class by what OSM thinks
  each footprint is.

### Worth looking at specifically

1. **`rescued = true`** — the buildings that are here only because of a POI.
   With `rescued_by = 'poi'` each should show a shop, a practice, a restaurant
   or a stable inside a residential-coded, farm or parking polygon; one that
   does not is a misplaced POI. With `rescued_by = 'site'` each should be a
   main building of the facility around it — a care-home wing, a riding hall, a
   campus building — not a shed; a shed here means the threshold is too low.
2. **`drop_reason = 'class'`** — the 40 classes dropped outright, the canopies
   above all: 94,571 of them, median 10 m² and 3.9 m. Tick classes on and off in
   the legend to see each one.
3. **`function = '31001_2000' AND area_m2 < 50`** — the shed question, still
   open. 369,675 buildings are tagged "business or commerce" with a median of
   28.7 m².
4. **`drop_reason = 'home_no_poi'`** — should be housing and nothing else.
5. **`source = 'osm'`** — the fill. Every magenta footprint should sit in a
   visible hole in the ALKIS layer; one drawn on top of an ALKIS building means
   the coverage threshold let a duplicate through.


In [9]:
EXPERIMENTAL_DIR.mkdir(parents=True, exist_ok=True)

# Columns that make the review easy without writing filter expressions.
# `aaa_class`, `kept`, `drop_reason`, `rescued` and `n_poi` come from section 7.
_n = slim['function'].value_counts()
slim['class_label'] = (
    slim['function'] + ' \u00b7 ' + slim['label_en'].fillna('?')
    + ' \u00b7 ' + slim['function'].map(_n).map('{:,}'.format))

inspect_cols = ['building_id', 'source', 'alkis_id', 'function', 'aaa_class',
                'class_label', 'label_en', 'activities', 'n_activities',
                'kept', 'drop_reason', 'rescued', 'rescued_by', 'n_poi', 'in_site',
                'area_m2', 'volume_3d_m3', 'volume_old_m3', 'volume_ratio',
                'height_top_max_m', 'roof_shape', 'name', 'address', 'city',
                'n_parts', 'osm_building', 'osm_levels', 'osm_height_m', 'geometry']
insp = slim[inspect_cols]

if LABELLED_INSPECT_FILE.exists():
    try:
        LABELLED_INSPECT_FILE.unlink()
    except PermissionError as e:
        raise RuntimeError(
            f'{LABELLED_INSPECT_FILE.name} is locked - close it in QGIS and rerun '
            f'this cell. Original error: {e}'
        ) from None

print(f'Writing {len(insp):,} rows x {len(insp.columns)} columns to '
      f'{LABELLED_INSPECT_FILE.name} ...', flush=True)
t0 = time.perf_counter()
insp.to_file(LABELLED_INSPECT_FILE, layer='buildings', driver='GPKG')
print(f'  ok  {LABELLED_INSPECT_FILE.stat().st_size / 1e6:,.1f} MB  '
      f'[{time.perf_counter() - t0:,.0f}s]')

chk = gpd.read_file(LABELLED_INSPECT_FILE, layer='buildings', rows=4)
print(f'  ok  read back: CRS {chk.crs}, {len(chk.columns)} columns')
print(chk[['function', 'label_en', 'activities', 'kept', 'drop_reason', 'n_poi',
           'area_m2', 'height_top_max_m']].to_string(index=False))
_chk_osm = gpd.read_file(LABELLED_INSPECT_FILE, layer='buildings',
                         where="source = 'osm'", rows=3)
print(f'  ok  read back {len(_chk_osm)} OSM rows by `source` filter:')
print(_chk_osm[['building_id', 'function', 'osm_building', 'kept', 'area_m2']]
      .to_string(index=False))

Writing 916,580 rows x 29 columns to 04_buildings_labelled.gpkg ...


  ok  490.3 MB  [12s]
  ok  read back: CRS EPSG:25832, 29 columns
  function                           label_en    activities  kept drop_reason  n_poi  area_m2  height_top_max_m
51002_1250                               mast          work False       class      0     4.00             3.500
51002_1250                               mast          work False       class      0     4.00             3.500
31001_1000              residential buildings   home;meetup False home_no_poi      0   115.68             7.404
31001_2000 Buildings for business or commerce work;business  True         NaN      0   212.80             4.377


  ok  read back 3 OSM rows by `source` filter:
 building_id function osm_building  kept  area_m2
way/22943903      OSM          yes  True    73.30
way/23862345      OSM   commercial  True   318.54
way/24989328      OSM          yes  True    97.75


In [10]:
# --- write a QGIS style so the layer opens already categorised --------------
# Hand-written QML rather than exported from QGIS, because QGIS is not scriptable
# from here. Uses the <prop k=.../> form, which QGIS 3.x still accepts, and is
# belt-and-braces: if a QGIS version refuses the file, `class_label` still gives
# a three-click path to the same categorised view.
#
# Colours carry meaning rather than being arbitrary: the family a code belongs to
# decides the hue, so the map is readable before any styling is touched.
_FAMILY_RAMPS = {
    '1': [(31, 120, 180), (66, 146, 198), (107, 174, 214), (158, 202, 225),
          (198, 219, 239), (222, 235, 247)],                      # residential: blues
    '2': [(227, 74, 51), (239, 101, 72), (252, 141, 89), (253, 187, 132),
          (253, 212, 158), (254, 232, 200)],                      # business: oranges/reds
    '3': [(35, 132, 67), (65, 171, 93), (120, 198, 121), (173, 221, 142),
          (217, 240, 163), (247, 252, 185)],                      # public: greens
}
_GREYS = [(82, 82, 82), (115, 115, 115), (150, 150, 150), (189, 189, 189),
          (217, 217, 217)]
_OSM_FILL_COLOUR = (231, 41, 138)                                 # OSM fill: magenta


def _colour(code, i):
    if code == OSM_GAP_FUNCTION_CODE:
        return _OSM_FILL_COLOUR
    aaa, gfk = code.split('_', 1)
    if aaa != '31001':
        return _GREYS[i % len(_GREYS)]
    ramp = _FAMILY_RAMPS.get(gfk[0], _GREYS)
    return ramp[i % len(ramp)]


def _esc(t):
    return (str(t).replace('&', '&amp;').replace('<', '&lt;')
            .replace('>', '&gt;').replace('"', '&quot;'))


# Biggest classes first, so the legend reads in order of importance.
order = slim['function'].value_counts().index.tolist()
labels = (slim.drop_duplicates('function').set_index('function')['class_label'])

cats, syms, seen = [], [], {}
for n, code in enumerate(order):
    if code == OSM_GAP_FUNCTION_CODE:
        fam = 'osm'
    elif code.startswith('31001'):
        fam = code.split('_', 1)[1][0]
    else:
        fam = 'x'
    seen[fam] = seen.get(fam, 0)
    r, g, b = _colour(code, seen[fam])
    seen[fam] += 1
    cats.append(f'   <category value="{_esc(code)}" symbol="{n}" '
                f'label="{_esc(labels.get(code, code))}" render="true"/>')
    syms.append(
        f'   <symbol type="fill" name="{n}" alpha="1" clip_to_extent="1" force_rhr="0">\n'
        f'    <layer class="SimpleFill" enabled="1" pass="0" locked="0">\n'
        f'     <prop k="color" v="{r},{g},{b},255"/>\n'
        f'     <prop k="style" v="solid"/>\n'
        f'     <prop k="outline_color" v="35,35,35,180"/>\n'
        f'     <prop k="outline_style" v="solid"/>\n'
        f'     <prop k="outline_width" v="0.04"/>\n'
        f'     <prop k="outline_width_unit" v="MM"/>\n'
        f'     <prop k="joinstyle" v="bevel"/>\n'
        f'    </layer>\n'
        f'   </symbol>')

qml = (
    "<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>\n"
    '<qgis version="3.28.0" styleCategories="Symbology">\n'
    ' <renderer-v2 type="categorizedSymbol" attr="function" forceraster="0"\n'
    '              symbollevels="0" enableorderby="0" referencescale="-1">\n'
    '  <categories>\n' + '\n'.join(cats) + '\n  </categories>\n'
    '  <symbols>\n' + '\n'.join(syms) + '\n  </symbols>\n'
    ' </renderer-v2>\n'
    ' <blendMode>0</blendMode>\n'
    '</qgis>\n'
)

qml_path = LABELLED_INSPECT_FILE.with_suffix('.qml')
qml_path.write_text(qml, encoding='utf-8')

# Parse it back - a malformed QML fails silently in QGIS, which is worse than
# an error here.
import xml.etree.ElementTree as _ET
_root = _ET.fromstring(qml)
_ncat = len(_root.findall('.//category'))
_nsym = len(_root.findall('.//symbol'))
if _ncat != len(order) or _nsym != len(order):
    raise AssertionError(f'QML has {_ncat} categories and {_nsym} symbols, '
                         f'expected {len(order)} of each')
print(f'  ok  {qml_path.name}: {_ncat} categories, valid XML, '
      f'{qml_path.stat().st_size / 1e3:,.0f} KB')
print()
print('  ..  in QGIS: add 04_buildings_labelled.gpkg - the .qml loads with it,')
print('      and every class appears in the legend with its own tick box;')
print(f'      the OSM fill is the magenta entry "{labels.get(OSM_GAP_FUNCTION_CODE, OSM_GAP_FUNCTION_CODE)}".')
print()
print('  ..  the legend, biggest first:')
for code in order[:12]:
    print(f'        {labels.get(code, code)}')
print(f'        ... and {len(order) - 12} more')

  ok  04_buildings_labelled.qml: 89 categories, valid XML, 51 KB

  ..  in QGIS: add 04_buildings_labelled.gpkg - the .qml loads with it,
      and every class appears in the legend with its own tick box;
      the OSM fill is the magenta entry "OSM · Building mapped in OSM, no ALKIS record · 47,264".

  ..  the legend, biggest first:
        31001_2000 · Buildings for business or commerce · 369,675
        31001_1000 · residential buildings · 327,264
        51009_1610 · canopy · 94,571
        OSM · Building mapped in OSM, no ALKIS record · 47,264
        31001_2720 · Agricultural and forestry business building · 18,850
        31001_2100 · Commercial and industrial buildings · 11,849
        31001_2010 · Buildings for trade and services · 8,631
        31001_2500 · Buildings for supply · 7,806
        31001_1120 · Residential buildings with trade and services · 5,793
        31001_1210 · Agricultural and forestry residential building · 4,197
        31001_3000 · Buildings for public

## 9. Drop what is not a place of activity

A capacity model places demand that happens *at* a building — work, shopping,
errands, education, leisure. Section 7 marked three kinds of row that host none
of it; this section reports each rule and removes what it selected.

### Rule 1 — structures and classes dropped outright

`ALKIS_DROP_ALWAYS` in `config.py`, one line of reasoning per code. The bulk is
the 94,571 canopies — forecourt roofs, carports, bus shelters — followed by
silos, masts, solar arrays, wind turbines, tanks and chimneys. Also here, by
decision rather than by physics: supply and disposal buildings (median 14 m²
transformer boxes), greenhouses, the five transport-operations classes,
barracks, mines, and the tourism structures — towers, ruins, mills, stands,
shelters, huts.

**No POI rescue for this list, and the POIs inside it are ignored.** Measured
before the decision: 141 POIs sit under canopies — Aral, Esso, Shell and Jet
forecourts, a Lidl, two Sparkasse branches, three pharmacies — because the
mapper put the point under the roof. Decided 2026-09-10: they are not re-homed
in 04.5 either. The petrol station's shop is a `31001_2130` building of its own
and keeps its `errands;work`; a pharmacy or bank whose node sits under the
entrance canopy loses that one POI, and the building next to it is judged on
its own class. The count printed below is what that costs.

### Rule 2 — classes dropped unless a POI sits inside

`ALKIS_DROP_UNLESS_POI`: parking garages and decks, where an Aldi, a KiK, a
bakery and two gyms occupy the ground floor of a few, and agricultural business
buildings, where 95 of 18,850 hold a riding stable, a farm shop or a café. The
POI keeps the building; the rest go.

### Rule 3 — homes, unless a POI sits inside

**Expressed as a rule over `activities`, not as a code list.** A flat with a
shop on the ground floor carries `home;meetup;work;shopping;business;errands`,
which is not a subset of `{home, meetup}`, so it survives without anyone
maintaining an exception list. The rule's expected result is declared in
`config.py` and **asserted against the data** — if the computed set differs, the
activity map or the source release has changed and the reasoning needs
revisiting rather than the list being quietly edited.

Then the rescue. ALKIS codes a building by its dominant use, so the corner
restaurant, the hairdresser and the doctor's practice in a residential block are
coded `Wohngebäude` too — and so are care homes, student halls and the staff
buildings of a campus. Where step 01 found a point or footprint POI inside such
a building it stays, flagged `rescued_by = 'poi'`; where the building is a
substantial one inside a site polygon it stays as `rescued_by = 'site'`. The
same correction the previous pipeline made, with the size guard section 7
explains.

### The OSM fill passes through untouched

The rows section 6 added have no `activities` and no AdV code, so no rule can
judge them and they are all kept — including the ones OSM tags `house` or
`apartments`. That is reported below rather than hidden. Deciding what an OSM
`building=*` tag means in activity terms is a reference table nobody has written
yet, and it is not decided implicitly here.

### What this costs, decided deliberately

**`meetup` is a redistribution target.** The original pipeline maps MiD
`meetup → Leisure`, and dropping `31001_1000` removes about 300M m³ of it. So
visiting-friends trips have nowhere to land, and Leisure demand falls entirely
on pubs, sports halls and cinemas.

Accepted: home visits are out of scope for a capacity model. Worth revisiting
here if the Leisure totals later look implausibly concentrated on venues.


In [11]:
# --- 9a. the home-only rule must still select exactly the codes config expects
found = set(slim.loc[slim['home_only'], 'function'].unique())
expected = set(ALKIS_EXPECTED_HOME_ONLY)
if found != expected:
    raise AssertionError(
        'the home-only rule selects a different set of codes than '
        'ALKIS_EXPECTED_HOME_ONLY in config.py.'
        f' only in data: {sorted(found - expected)};'
        f' only in config: {sorted(expected - found)}.'
        ' The activity map or the source release has changed - review the '
        'reasoning in config.py before editing the list.'
    )
print(f'  ok  {len(found)} home-only codes, matching config.py exactly')


def _mm3(v):
    return v.sum() / 1e6


# --- 9b. rule 1: dropped outright, with the POIs 04.5 has to re-home ----------
print()
print('RULE 1 - dropped outright (ALKIS_DROP_ALWAYS), biggest first:')
_a = (slim[slim['drop_reason'] == 'class']
      .groupby(['function', 'label_en'])
      .agg(n=('building_id', 'size'), vol=('volume_3d_m3', _mm3), pois=('n_poi', 'sum'))
      .reset_index().sort_values('n', ascending=False))
for _, r in _a.iterrows():
    print(f'  {r["function"]:<12} {r["label_en"][:38]:<38} {int(r["n"]):>7,} bld  '
          f'{r["vol"]:>6.2f}M m3  {int(r["pois"]):>4} POIs inside')
_absent = sorted(set(ALKIS_DROP_ALWAYS) - set(_a['function']))
print(f'  {len(_a)} classes, {int(_a["n"].sum()):,} buildings, {_a["vol"].sum():.1f}M m3'
      + (f'; {len(_absent)} listed codes do not occur in this region: {_absent}' if _absent else ''))
_under = poi_hits[poi_hits['function'].isin(ALKIS_DROP_ALWAYS)]
print(f'  ..  {len(_under):,} POIs sit inside these - ignored by decision, not re-homed '
      'in 04.5. By use: '
      + ', '.join(f'{k} {v}' for k, v in _under['poi_use'].value_counts().head(8).items()))

# --- 9c. rule 2: dropped unless a POI sits inside -----------------------------
print()
print('RULE 2 - dropped unless a POI sits inside (ALKIS_DROP_UNLESS_POI):')
for _code in ALKIS_DROP_UNLESS_POI:
    _g = slim[slim['function'] == _code]
    if _g.empty:
        print(f'  {_code:<12} not in this region')
        continue
    _k = _g[_g['kept']]
    print(f'  {_code:<12} {_g["label_en"].iloc[0][:38]:<38} {len(_g):>7,} bld -> '
          f'{len(_k):>5,} kept ({int((_k["rescued_by"] == "poi").sum()):,} by a POI inside, '
          f'{int((_k["rescued_by"] == "site").sum()):,} by a site), {len(_g) - len(_k):,} dropped '
          f'({_mm3(_g["volume_3d_m3"]) - _mm3(_k["volume_3d_m3"]):.2f}M m3)')
    _u = poi_hits.loc[poi_hits['function'] == _code, 'poi_use'].value_counts().head(6)
    print(f'  {"":<12} POIs that kept them: ' + ', '.join(f'{a} {b}' for a, b in _u.items()))

# --- 9d. rule 3: homes, unless a POI sits inside ------------------------------
print()
print('RULE 3 - activities are a subset of {' + ', '.join(sorted(ALKIS_HOME_ONLY_ACTIVITIES))
      + '}, unless a POI sits inside:')
for _code in sorted(found):
    _g = slim[(slim['function'] == _code) & slim['home_only']]
    _k = _g[_g['kept']]
    print(f'  {_code:<12} {_g["label_en"].iloc[0][:38]:<38} {len(_g):>7,} bld -> '
          f'{len(_k):>5,} kept ({int((_k["rescued_by"] == "poi").sum()):,} by a POI inside, '
          f'{int((_k["rescued_by"] == "site").sum()):,} by a site), {len(_g) - len(_k):,} dropped '
          f'({_mm3(_g["volume_3d_m3"]) - _mm3(_k["volume_3d_m3"]):.1f}M m3)')
    print(f'  {"":<12} {ALKIS_EXPECTED_HOME_ONLY[_code]}')
_u = poi_hits.loc[poi_hits['function'].isin(found), 'poi_use'].value_counts()
print(f'  ..  POIs inside the rescued homes ({int(_u.sum()):,}): '
      + ', '.join(f'{a} {b}' for a, b in _u.head(12).items()))
_sr = slim[slim['home_only'] & (slim['rescued_by'] == 'site')]
_su = site_hits.loc[site_hits['building_id'].isin(_sr['building_id']), 'poi_use'].value_counts()
print(f'  ..  homes kept by a site polygon alone ({len(_sr):,}, median {_sr["area_m2"].median():,.0f} m2): '
      + ', '.join(f'{a} {b}' for a, b in _su.head(10).items()))

# Nothing carrying a non-home activity may have gone through rule 3. This is the
# check that the "flat with a shop downstairs" case actually survived.
_gone = slim[slim['drop_reason'] == 'home_no_poi']
_leak = _gone[_gone['activities'].fillna('').str.contains(
    'work|business|shopping|errands|education|lessons|sports|daycare|leisure',
    regex=True)]
if len(_leak):
    raise AssertionError(
        f'{len(_leak):,} buildings dropped as home-only carry a non-home activity: '
        f'{_leak["function"].unique().tolist()}'
    )
print('  ok  nothing dropped as home-only carries work, shopping, errands, education or leisure')

# --- 9e. apply ----------------------------------------------------------------
n_before = len(slim)
v_before = slim['volume_3d_m3'].sum()
enriched = (slim[slim['kept']]
            .drop(columns=['home_only', 'has_poi', 'n_poi', 'in_site', 'drop_reason', 'kept'])
            .reset_index(drop=True))
require_unique(enriched, 'building_id', 'enriched buildings')
print()
print(f'  ..  buildings : {n_before:,} -> {len(enriched):,} '
      f'({n_before - len(enriched):,} dropped, '
      f'{100 * (n_before - len(enriched)) / n_before:.2f} %)')
print(f'  ..  volume    : {v_before / 1e6:,.1f}M -> '
      f'{enriched["volume_3d_m3"].sum() / 1e6:,.1f}M m3 '
      f'({100 * (1 - enriched["volume_3d_m3"].sum() / v_before):.2f} % removed)')
for _src in ('alkis', 'osm'):
    print(f'        {_src:<6} {int((slim["source"] == _src).sum()):>8,} -> '
          f'{int((enriched["source"] == _src).sum()):>8,}')
print(f'  ..  rescued   : {int(enriched["rescued"].sum()):,} buildings are here only because of '
      f'a POI - {int((enriched["rescued_by"] == "poi").sum()):,} with one inside, '
      f'{int((enriched["rescued_by"] == "site").sum()):,} inside a site polygon '
      '(`rescued`, `rescued_by` carried forward)')

_osm_rows = slim[slim['kept'] & (slim['source'] == 'osm')]
_osm_home_tags = _osm_rows['osm_building'].isin(
    ['house', 'detached', 'semidetached_house', 'apartments', 'residential',
     'terrace', 'bungalow', 'dormitory']).sum()
print(f'  ..  OSM fill : {len(_osm_rows):,} rows carried through untouched - no AdV code '
      'and no activities, so no rule can judge them')
print(f'               {int(_osm_home_tags):,} of them carry a residential building=* tag and '
      f'would go if an OSM activity map existed; {int(_osm_rows["has_poi"].sum()):,} hold a POI')

print()
print('  ..  what the enriched layer now looks like, by activity:')
_ex = (enriched[['building_id', 'volume_3d_m3']]
       .assign(a=enriched['activities'].fillna('').str.split(ALKIS_ACTIVITY_SEP))
       .explode('a'))
_ex['a'] = _ex['a'].str.strip()
_ex = _ex[_ex['a'] != '']
_t2 = _ex.groupby('a').agg(buildings=('building_id', 'size'),
                           volume_Mm3=('volume_3d_m3', lambda v: round(v.sum() / 1e6, 1)))
_t2['pct_vol'] = (100 * _t2['volume_Mm3'] /
                  (enriched['volume_3d_m3'].sum() / 1e6)).round(1)
print(_t2.sort_values('buildings', ascending=False).to_string())


  ok  2 home-only codes, matching config.py exactly

RULE 1 - dropped outright (ALKIS_DROP_ALWAYS), biggest first:
  51009_1610   canopy                                  94,571 bld   10.10M m3   138 POIs inside
  31001_2500   Buildings for supply                     7,806 bld    3.16M m3    12 POIs inside
  51003_1201   silo                                     1,784 bld    2.44M m3     1 POIs inside
  51002_1250   mast                                     1,772 bld    0.47M m3     0 POIs inside
  51002_1230   Solar cells                              1,177 bld    0.65M m3     0 POIs inside
  31001_2600   Building for disposal                      770 bld    1.05M m3     4 POIs inside
  31001_2740   greenhouse, greenhouse                     602 bld    0.75M m3    13 POIs inside
  51002_1220   Wind turbine                               423 bld    0.87M m3     0 POIs inside
  31001_2410   Operational building for road traffic      363 bld    0.33M m3     5 POIs inside
  51003_1205   tank  

  31001_2461   Parking garage                              65 bld ->     5 kept (5 by a POI inside, 0 by a site), 60 dropped (0.95M m3)
               POIs that kept them: fitness_centre 2, bakery 1, supermarket 1, biergarten 1, nightclub 1, vacant 1


  31001_2462   Parking deck                                80 bld ->     3 kept (1 by a POI inside, 2 by a site), 77 dropped (0.74M m3)
               POIs that kept them: linux 1, community_centre 1
  31001_2720   Agricultural and forestry business bui  18,850 bld ->   126 kept (91 by a POI inside, 35 by a site), 18,724 dropped (28.07M m3)
               POIs that kept them: equestrian 21, farm 14, cafe 6, florist 5, veterinary 4, building 4

RULE 3 - activities are a subset of {home, meetup}, unless a POI sits inside:


  31001_1000   residential buildings                  327,264 bld -> 4,300 kept (4,023 by a POI inside, 277 by a site), 322,964 dropped (283.7M m3)
               residential buildings (home;meetup) - 327,264 buildings, 299.9M m3
  31001_1210   Agricultural and forestry residential    4,197 bld ->    48 kept (40 by a POI inside, 8 by a site), 4,149 dropped (6.0M m3)
               agricultural and forestry residential (home) - 4,197 buildings, 6.2M m3
  ..  POIs inside the rescued homes (4,904): restaurant 294, chalet 232, hairdresser 202, social_facility 192, doctors 182, fast_food 159, apartment 136, bakery 128, dentist 114, cafe 105, guest_house 90, beauty 90
  ..  homes kept by a site polygon alone (285, median 363 m2): social_facility 78, school 29, university 22, hospital 20, factory 16, research_institute 15, chalet 13, sports_centre 12, camp_site 7, swimming 6


  ok  nothing dropped as home-only carries work, shopping, errands, education or leisure


  ok  enriched buildings.building_id: unique and non-null (459,932)

  ..  buildings : 916,580 -> 459,932 (456,648 dropped, 49.82 %)
  ..  volume    : 677.3M -> 332.5M m3 (50.90 % removed)
        alkis   869,316 ->  412,668
        osm      47,264 ->   47,264
  ..  rescued   : 4,482 buildings are here only because of a POI - 4,160 with one inside, 322 inside a site polygon (`rescued`, `rescued_by` carried forward)


  ..  OSM fill : 47,264 rows carried through untouched - no AdV code and no activities, so no rule can judge them
               6,149 of them carry a residential building=* tag and would go if an OSM activity map existed; 689 hold a POI

  ..  what the enriched layer now looks like, by activity:


                 buildings  volume_Mm3  pct_vol
a                                              
work                407323       312.6     94.0
business            402182       288.3     86.7
errands              18849        61.2     18.4
shopping             15304        49.8     15.0
meetup               15070        40.6     12.2
home                 12120        34.5     10.4
leisure               7606        20.9      6.3
education             2710        17.3      5.2
lessons               1597         2.1      0.6
sports                 638         4.6      1.4
early_education        186         1.0      0.3
other                   94         0.0      0.0


## 10. Where this leaves us

`enriched` holds **459,932 buildings**: 412,668 from ALKIS — every one a place
where something other than living happens, or a home a POI proved otherwise —
with `label_en`, `activities`, `n_activities`, `rescued` and `rescued_by`
attached, **plus the 47,264 OSM footprints ALKIS does not have**, with
`source = 'osm'`, no volume and no activities. Still in memory; the layer is
written once the POI join is in.

| | buildings | volume |
|---|---|---|
| in from step 03, plus the OSM fill | 916,580 | 677.3M m³ |
| rule 1 — 40 classes dropped outright | −110,674 | −25.3M m³ |
| rule 2 — 3 classes, no POI inside | −18,861 | −29.8M m³ |
| rule 3 — homes, no POI inside | −327,113 | −289.7M m³ |
| **out** | **459,932** | **332.5M m³** |

4,482 of the survivors are there only because of a POI: 4,160 with a point or
footprint POI inside — restaurants, hairdressers, doctors, bakeries, dentists,
and 232 holiday chalets — and 322 as substantial buildings inside a site
polygon, mostly care homes, schools, campuses and hospitals coded residential.

One review file is on disk:

* **`04_buildings_labelled.gpkg`** — every building including the dropped ones,
  with `kept`, `drop_reason`, `rescued`, `rescued_by`, `n_poi` and `in_site` so
  every decision is visible on the map, and the OSM fill as its own magenta
  legend class

### Still open

* **`31001_2000` "buildings for business or commerce" — 369,675 buildings,
  42.5 % of the layer, tagged `work;business`.** Median area 28.7 m², median
  height 2.9 m, 74 % flat-roofed. That is a garden shed or a garage, not a
  commercial premises. It is 15.2 % of volume against 42.5 % of count, so volume
  weighting limits the harm, but 370,000 sheds carrying `work` is the largest
  single question left in this layer. A size rule, not a class rule — the
  summary in section 5 is the evidence for it.
* **233 POIs sit inside buildings rule 1 dropped, and are ignored** — 138
  under canopies (fuel stations, pharmacies, banks), 13 in greenhouses (garden
  centres), the rest in supply, transport and tourism structures. Decided: not
  re-homed in 04.5. Snapping is only for building-bound POIs that lie outside
  every footprint.
* **Tourism accommodation rescues homes.** 232 `chalet`, 136 `apartment` and
  90 `guest_house` POIs keep residential-coded holiday homes. Whether a holiday
  flat is an activity destination is for the classification, not for this
  filter — but it is where those buildings come from.
* **The site threshold is a judgement.** `POI_SITE_RESCUE_MIN_AREA_M2 = 200`
  keeps 322 buildings; 100 m² would keep 548, 50 m² 795. Section 7 prints the
  ladder on every run.
* **The OSM fill has no volume and no activities.** On the map it closes the
  holes; in a volume-weighted redistribution it currently weighs nothing, and no
  rule can see the 6,149 houses among it. Both need a decision: a volume
  estimate (OSM has `building:levels` on ~5 % of them) and an OSM
  `building=*` → activity map. 689 of them hold a POI.

### Next: 04.5, the OSM POI join

`01_all_pois.gpkg` joined by `poi_role` — points by containment, footprints 1:1,
sites onto every contained building, and building-bound POIs outside every
footprint to the nearest kept building within the snap radius. That is where the nested shopping-centre units come
in.

**Caveat carried forward:** notebook 01 has never been tested, 04.5 depends
entirely on its POI layer, and its `poi_role` has no concept of a POI nested
inside another POI — which is exactly what a shopping centre is. Worth verifying
before the join is built on top of it.
